In [4]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel

In [5]:
# Load the speed dating data
df = pd.read_csv(
    "Speed_Dating_Data.csv",
    encoding="latin-1"
)

df.head()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [6]:
# Keep only the variables used in Q3 / Q4
data = df[['iid', 'pid', 'dec', 'attr', 'intel', 'fun']].copy()

# Drop rows with missing values in the relevant columns
data = data.dropna(subset=['dec', 'attr', 'intel', 'fun'])

data.head()

,iid,pid,dec,attr,intel,fun
0,1,11.0,1,6.0,7.0,7.0
1,1,12.0,1,7.0,7.0,8.0
2,1,13.0,1,5.0,9.0,8.0
3,1,14.0,1,7.0,8.0,7.0
4,1,15.0,1,5.0,7.0,7.0


In [7]:
# Use first 90% as training and last 10% as test
data_test = data[7157:].copy()
data_train = data[:7157].copy()

print("Training size:", data_train.shape[0])
print("Test size:", data_test.shape[0])

Training size: 7157
Test size: 795


In [8]:
cols = ['attr', 'intel', 'fun']

train_means = data_train[cols].mean()
train_stds = data_train[cols].std()

data_train[cols] = (data_train[cols] - train_means) / train_stds
data_test[cols] = (data_test[cols] - train_means) / train_stds

data_train.head()

,iid,pid,dec,attr,intel,fun
0,1,11.0,1,-0.122467,-0.265137,0.290312
1,1,12.0,1,0.394062,-0.265137,0.802330
2,1,13.0,1,-0.638996,1.040615,0.802330
3,1,14.0,1,0.394062,0.387739,0.290312
4,1,15.0,1,-0.638996,-0.265137,0.290312


In [9]:
data_dict_q4 = {
    'N': data_train.shape[0],
    'y': data_train['dec'].astype(int).tolist(),
    'attr': data_train['attr'].astype(float).tolist(),
    'intel': data_train['intel'].astype(float).tolist(),
    'fun': data_train['fun'].astype(float).tolist(),

    'M': data_test.shape[0],
    'attr_test': data_test['attr'].astype(float).tolist(),
    'intel_test': data_test['intel'].astype(float).tolist(),
    'fun_test': data_test['fun'].astype(float).tolist(),
}

# Actual test labels stay in Python for evaluation
y_test = data_test['dec'].astype(int).to_numpy()

print("First 5 test labels:", y_test[:5])

First 5 test labels: [1 1 0 1 0]


In [11]:
model_q4 = CmdStanModel(stan_file="q4.stan")

fit_q4 = model_q4.sample(
    data=data_dict_q4,
    chains=4,
    iter_warmup=1000,
    iter_sampling=2500,
    seed=42
)

17:36:58 - cmdstanpy - INFO - compiling stan file /home/crispynacho/cogmod/in_class/cog-modeling-class-nc-ks/HW4/q4.stan to exe file /home/crispynacho/cogmod/in_class/cog-modeling-class-nc-ks/HW4/q4
17:37:11 - cmdstanpy - INFO - compiled model executable: /home/crispynacho/cogmod/in_class/cog-modeling-class-nc-ks/HW4/q4
17:37:11 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]


chain 1:   3%|▎         | 100/3500 [00:01<00:57, 59.56it/s, (Warmup)]





chain 1:   9%|▊         | 300/3500 [00:06<01:08, 46.55it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:08<01:06, 46.29it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:11<01:08, 43.79it/s, (Warmup)]


chain 1:  17%|█▋        | 600/3500 [00:13<01:04, 44.82it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:14<00:56, 49.30it/s, (Warmup)]

chain 1:  23%|██▎       | 800/3500 [00:15<00:47, 56.40it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:17<00:42, 60.89i


17:38:19 - cmdstanpy - INFO - CmdStan done processing.


In [12]:
print(fit_q4.diagnose())

summary_q4 = fit_q4.summary()

print("Max R_hat:", summary_q4['R_hat'].max())
print("Min ESS_bulk:", summary_q4['ESS_bulk'].min())
print("Min ESS_tail:", summary_q4['ESS_tail'].min())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.

Max R_hat: 1.00133
Min ESS_bulk: 4913.15
Min ESS_tail: 5000.0


In [13]:
draws_q4 = fit_q4.draws_pd()

p_cols = [col for col in draws_q4.columns if col.startswith("p_test[")]
p_draws = draws_q4[p_cols].to_numpy()

print("Shape of predictive draw matrix:", p_draws.shape)

Shape of predictive draw matrix: (10000, 795)


In [14]:
p_bar = p_draws.mean(axis=0)

print("First 10 predictive means:", p_bar[:10])

First 10 predictive means: [0.842564   0.90406075 0.75563335 0.51029961 0.06109579 0.06998132
 0.73634498 0.0952417  0.51491303 0.88844548]


In [15]:
brier = np.mean((p_bar - y_test) ** 2)
print("Brier score:", brier)

Brier score: 0.16969949403280243


In [16]:
y_pred = (p_bar >= 0.5).astype(int)
accuracy = np.mean(y_pred == y_test)

print("Classification accuracy:", accuracy)

Classification accuracy: 0.7647798742138365


In [17]:
majority_class = int(data_train['dec'].mean() >= 0.5)
baseline_pred = np.full_like(y_test, majority_class)

baseline_acc = np.mean(baseline_pred == y_test)
baseline_brier = np.mean((baseline_pred - y_test) ** 2)

print("Baseline accuracy:", baseline_acc)
print("Baseline Brier:", baseline_brier)

Baseline accuracy: 0.6075471698113207
Baseline Brier: 0.39245283018867927


In [18]:
brier_draws = np.mean((p_draws - y_test[None, :]) ** 2, axis=1)
acc_draws = np.mean((p_draws >= 0.5) == y_test[None, :], axis=1)

print("Posterior mean Brier:", brier_draws.mean())
print("95% interval for Brier:", np.percentile(brier_draws, [2.5, 97.5]))

print("Posterior mean accuracy:", acc_draws.mean())
print("95% interval for accuracy:", np.percentile(acc_draws, [2.5, 97.5]))

Posterior mean Brier: 0.16980466174140316
95% interval for Brier: [0.16890102 0.17087331]
Posterior mean accuracy: 0.7642106918238994
95% interval for accuracy: [0.75849057 0.76855346]
